In [1]:
import numpy as np

images = np.load(
    "../data/processed/images.npy"
)

labels = np.load(
    "../data/processed/labels.npy"
)

print(images.shape)
print(labels.shape)

(41167, 32, 32, 3)
(41167,)


In [2]:
mask = labels != -1

images_train = images[mask]
labels_train = labels[mask]

In [3]:
import torch
from torch.utils.data import Dataset
from PIL import Image

In [4]:
class CIFARDataset(Dataset):

    def __init__(
        self,
        images,
        labels,
        transform=None
    ):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image = Image.fromarray(
            self.images[idx]
        )

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [5]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    images_train,
    labels_train,
    test_size=0.2,
    stratify=labels_train,
    random_state=42
)

In [7]:
train_dataset = CIFARDataset(
    X_train,
    y_train,
    transform
)

val_dataset = CIFARDataset(
    X_val,
    y_val,
    transform
)

In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

In [9]:
import torch.nn as nn
from torchvision.models import resnet18

In [10]:
model = resnet18(
    weights="IMAGENET1K_V1"
)

In [11]:
model.fc = nn.Linear(
    model.fc.in_features,
    10
)

In [12]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

model = model.to(device)

cuda


In [13]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [14]:
epochs = 5
for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images_batch, labels_batch in train_loader:

        images_batch = images_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        outputs = model(images_batch)

        loss = criterion(
            outputs,
            labels_batch
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}: {running_loss:.4f}"
    )

Epoch 1: 515.5989
Epoch 2: 411.2372
Epoch 3: 359.6715
Epoch 4: 321.0916
Epoch 5: 265.2949


In [15]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for images_batch, labels_batch in val_loader:

        images_batch = images_batch.to(device)
        labels_batch = labels_batch.to(device)

        outputs = model(images_batch)

        preds = outputs.argmax(1)

        correct += (
            preds == labels_batch
        ).sum().item()

        total += labels_batch.size(0)

accuracy = correct / total

print(
    f"Validation Accuracy: {accuracy:.4f}"
)

Validation Accuracy: 0.7586


In [16]:
from torch.nn.functional import softmax

In [17]:
full_dataset = CIFARDataset(
    images_train,
    labels_train,
    transform
)

full_loader = DataLoader(
    full_dataset,
    batch_size=64,
    shuffle=False
)

In [18]:
all_probs = []

model.eval()

with torch.no_grad():

    for images_batch, _ in full_loader:

        images_batch = images_batch.to(device)

        outputs = model(images_batch)

        probs = softmax(
            outputs,
            dim=1
        )

        all_probs.append(
            probs.cpu().numpy()
        )

In [19]:
pred_probs = np.concatenate(
    all_probs,
    axis=0
)

In [20]:
print(pred_probs.shape)

(40967, 10)


In [21]:
np.save(
    "../outputs/pred_probs.npy",
    pred_probs
)

np.save(
    "../outputs/train_labels.npy",
    labels_train
)

In [22]:
torch.save(
    model.state_dict(),
    "../outputs/resnet18.pth"
)

# Pipeline

Corrupted Dataset <br>
        ↓ <br>
ResNet18  <br>
        ↓ <br>
Predictions <br>
        ↓ <br>
Prediction Probabilities <br>
        ↓ <br>
Save <br>